# ICT-18b -- 4 classes Wolfram (I/II/III/IV) vs signature thermodynamique

**Suite du pli 2 EPIC Origami Wolfram (#19766).** Confrontation des 4 classes canoniques de Wolfram (I/II/III/IV) a la signature thermodynamique de `ict.time_arrow`.

**Statut epistemique** -- **Planifie** : carnet en construction (c.1160). Grain DEEP/research-code.

**Outils** :
- `ict.wolfram_step` : organe de l'automate 1-D de Wolfram (PR #19793, OPEN MERGEABLE, non merge au moment de l'ecriture). Version simplifiee in-notebook (`wolfram_step_inline`) pour permettre l'execution sans merge prealable.
- `ict.time_arrow` : module time_arrow, strate 5 d'ICT. `compare_real_reversed_reversibilized`, `sigma_real`, `dist_real_vs_reversibilized`, `agentivity_index`.
- 4 regles canoniques : 0 (classe I), 4 (II), 30 (III), 110 (IV). 250 comme 5e reference (classe III).

**Verdict attendu** : sigma croissante I < II < III ~ IV (les deux derniers ont la plus forte dissipation, mais III/IV restent indiscernables sur la densite brute ; voir Exercice 2 pour la discrimination spatiale).

Cibles pedagogiques : 3 exercices C.1 (stubs pass/print) + 1 exemple guide.

## Plan

- **Exercice 1** -- Generation des 4 trajectoires-types (densite en fonction du pas).
- **Exercice 2** -- Discrimination fine III vs IV (analyse spatiale, gliders).
- **Exercice 3** -- Signature thermodynamique par classe (sigma, dist, agentivity).
- **Exercice 4** -- Calibration de la mesure de densite (n_cells, seed).


In [1]:
import os
import sys

ICT_ROOT = os.path.abspath(".")
if ICT_ROOT not in sys.path:
    sys.path.insert(0, ICT_ROOT)

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ict.time_arrow (deja merge sur main)
try:
    from ict.time_arrow import (
        compare_real_reversed_reversibilized,
        discretise,
    )
    TIME_ARROW_OK = True
except ImportError as e:
    TIME_ARROW_OK = False
    print(f"[warn] ict.time_arrow non importable: {e}")
    def compare_real_reversed_reversibilized(*args, **kwargs):
        return {"sigma_real": 0.0, "dist": 0.0, "agentivity_index": 0.0}
    def discretise(traj, k=10):
        return np.zeros((k,))


# Version inline de wolfram_step (en attendant PR #19793 merge)
def wolfram_step_inline(state, rule_number):
    '''Une etape de l'automate 1-D de Wolfram (rayon 1, conditions au bord = 0).'''
    n = len(state)
    state_int = state.astype(int)
    pattern = (state_int[:-2] << 2) | (state_int[1:-1] << 1) | state_int[2:]
    new_int = (rule_number >> pattern) & 1
    new = np.zeros(n, dtype=np.int8)
    new[1:-1] = new_int.astype(np.int8)
    return new


def wolfram_trajectory(rule_number, n_cells=64, n_steps=500, seed=33):
    '''Trajectoire d'un automate 1-D de Wolfram sur n_steps pas.'''
    rng = np.random.default_rng(seed)
    state = rng.integers(0, 2, size=n_cells).astype(np.int8)
    trajectory = np.zeros((n_steps, n_cells), dtype=np.int8)
    trajectory[0] = state
    for t in range(1, n_steps):
        state = wolfram_step_inline(state, rule_number)
        trajectory[t] = state
    return trajectory


def density(trajectory):
    '''Densite de 1 a chaque pas : moyenne par ligne.'''
    return trajectory.mean(axis=1)


print(f"numpy {np.__version__}, ICT_ROOT={ICT_ROOT}")
print(f"time_arrow OK = {TIME_ARROW_OK}")
print(f"wolfram_step_inline OK")


[warn] ict.time_arrow non importable: No module named 'ict'
numpy 2.3.5, ICT_ROOT=D:\Dev\CoursIA-c1160-19796
time_arrow OK = False
wolfram_step_inline OK


## Exercice 1 -- Generation des 4 trajectoires-types

**Consigne** : pour chaque regle canonique (0/I, 4/II, 30/III, 110/IV) et une 5e de reference (250/III), generer la trajectoire sur 500 pas et tracer la densite.

**Verdict attendu** :
- Regle 0 (I, fixe) : densite = 0 apres 1 pas, ecart-type = 0.
- Regle 4 (II, periodique) : densite oscille entre 2 valeurs, ecart-type borne.
- Regle 30 (III, chaotique) : densite ~ 0.5, ecart-type non nul, signature chaotique.
- Regle 110 (IV, complexe) : densite ~ 0.5, ecart-type non nul, gliders visibles en Exo 2.
- Regle 250 (III, chaotique) : similaire a 30.

C.1 : stub a completer.


In [2]:
# Exercice 1 -- Generation des 4 trajectoires-types
RULES = {0: "I", 4: "II", 30: "III", 110: "IV", 250: "III-ref"}
DENSITIES = {}

for rule, classe in RULES.items():
    traj = wolfram_trajectory(rule, n_cells=64, n_steps=500, seed=33)
    rho = density(traj)
    DENSITIES[rule] = {
        "classe": classe,
        "rho_mean": float(rho.mean()),
        "rho_std": float(rho.std()),
        "rho_asymptotic_mean": float(rho[-100:].mean()),
        "rho_asymptotic_std": float(rho[-100:].std()),
    }
    print(f"Rule {rule:3d} (classe {classe:6s}): rho_mean={DENSITIES[rule]['rho_mean']:.4f}, rho_std={DENSITIES[rule]['rho_std']:.4f}, asymptotique = {DENSITIES[rule]['rho_asymptotic_mean']:.4f} +/- {DENSITIES[rule]['rho_asymptotic_std']:.4f}")


Rule   0 (classe I     ): rho_mean=0.0009, rho_std=0.0209, asymptotique = 0.0000 +/- 0.0000
Rule   4 (classe II    ): rho_mean=0.0633, rho_std=0.0181, asymptotique = 0.0625 +/- 0.0000
Rule  30 (classe III   ): rho_mean=0.4908, rho_std=0.0294, asymptotique = 0.4922 +/- 0.0078
Rule 110 (classe IV    ): rho_mean=0.6706, rho_std=0.0808, asymptotique = 0.7188 +/- 0.0000
Rule 250 (classe III-ref): rho_mean=0.9668, rho_std=0.0252, asymptotique = 0.9688 +/- 0.0000


## Exercice 2 -- Discrimination fine III vs IV (analyse spatiale)

**Consigne** : pour Rule 30 (chaotique) et Rule 110 (complexe), generer la matrice 2-D espace-temps (n_cells colonnes, n_steps lignes) sur 200 pas. Detecter la presence de gliders (composantes connexes stables se translatant en diagonale).

**Verdict attendu** :
- Rule 110 (classe IV) a des gliders.
- Rule 30 (classe III) n'en a pas.

C.1 : stub a completer (`pass`).


In [3]:
# Exercice 2 -- Discrimination fine III vs IV
def detect_gliders(spacetime_matrix, min_persistence=10):
    '''Detecte les gliders dans une matrice espace-temps.

    Un glider est une composante connexe de cellules '1' qui se translate
    en diagonale (meme vitesse en x et en y) sur au moins `min_persistence` pas.

    Parametres
    ----------
    spacetime_matrix : np.ndarray de forme (n_steps, n_cells)
    min_persistence : int, nombre minimum de pas pour considerer un glider

    Retour
    ------
    dict : {n_gliders: int, glider_velocities: list[float]}
    '''
    n_steps, n_cells = spacetime_matrix.shape
    # Implementation C.1 -- a completer par l'etudiant
    pass


for rule in [30, 110]:
    traj = wolfram_trajectory(rule, n_cells=128, n_steps=200, seed=33)
    result = detect_gliders(traj, min_persistence=10)
    if result is None:
        print(f"Rule {rule:3d}: detect_gliders pas encore implemente (C.1 stub)")
    else:
        print(f"Rule {rule:3d}: {result['n_gliders']} gliders, vitesses={result['glider_velocities']}")


Rule  30: detect_gliders pas encore implemente (C.1 stub)
Rule 110: detect_gliders pas encore implemente (C.1 stub)


## Exercice 3 -- Signature thermodynamique par classe

**Consigne** : pour chaque classe, appliquer `ict.time_arrow.compare_real_reversed_reversibilized(traj_densites_quantifiees)`. Discretiser la trajectoire de densite en k quantiles (methode `discretise` d'ICT-18). Mesurer `sigma_real`, `dist_real_vs_reversibilized`, `agentivity_index`.

**Verdict attendu** : `sigma` croissante I < II < III ~ IV.

C.1 : stub a completer.


In [4]:
# Exercice 3 -- Signature thermodynamique par classe
THERMO = {}

for rule, classe in RULES.items():
    traj = wolfram_trajectory(rule, n_cells=64, n_steps=500, seed=33)
    rho = density(traj)
    rho_disc = discretise(rho, k=10)
    res = compare_real_reversed_reversibilized(rho_disc)
    THERMO[rule] = res
    print(f"Rule {rule:3d} (classe {classe:6s}): sigma_real={res['sigma_real']:.4f}, dist={res['dist']:.4f}, agentivity={res['agentivity_index']:.4f}")


Rule   0 (classe I     ): sigma_real=0.0000, dist=0.0000, agentivity=0.0000
Rule   4 (classe II    ): sigma_real=0.0000, dist=0.0000, agentivity=0.0000
Rule  30 (classe III   ): sigma_real=0.0000, dist=0.0000, agentivity=0.0000
Rule 110 (classe IV    ): sigma_real=0.0000, dist=0.0000, agentivity=0.0000
Rule 250 (classe III-ref): sigma_real=0.0000, dist=0.0000, agentivity=0.0000


## Exercice 4 -- Calibration de la mesure de densite

**Consigne** : faire varier `n_cells in {16, 32, 64, 128}` et `seed in {1, 33, 99}`. Verifier la stabilite de `asymptotic_density` et `asymptotic_std` pour chaque classe.

**Verdict attendu** : III et IV sont robustes (densite ~ 0.5, std borne). I et II peuvent montrer une dependance au seed pour les petites cellules (transitoires longs).

C.1 : stub a completer.


In [5]:
# Exercice 4 -- Calibration de la mesure
CALIBRATION = []

for rule, classe in RULES.items():
    for n_cells in [16, 32, 64, 128]:
        for seed in [1, 33, 99]:
            traj = wolfram_trajectory(rule, n_cells=n_cells, n_steps=500, seed=seed)
            rho = density(traj)
            CALIBRATION.append({
                "rule": rule,
                "classe": classe,
                "n_cells": n_cells,
                "seed": seed,
                "asymp_mean": float(rho[-100:].mean()),
                "asymp_std": float(rho[-100:].std()),
            })

# Resume : ecart-type sur les seeds pour chaque (rule, n_cells)
import statistics
SUMMARY = {}
for c in CALIBRATION:
    key = (c["rule"], c["n_cells"])
    SUMMARY.setdefault(key, []).append(c["asymp_mean"])

for (rule, n_cells), means in sorted(SUMMARY.items()):
    classe = RULES[rule]
    if len(means) > 1:
        spread = max(means) - min(means)
    else:
        spread = 0.0
    print(f"Rule {rule:3d} (classe {classe:6s}), n_cells={n_cells:3d}: asymptotique = {statistics.mean(means):.4f} +/- {statistics.stdev(means) if len(means) > 1 else 0:.4f} (spread sur seeds = {spread:.4f})")


Rule   0 (classe I     ), n_cells= 16: asymptotique = 0.0000 +/- 0.0000 (spread sur seeds = 0.0000)
Rule   0 (classe I     ), n_cells= 32: asymptotique = 0.0000 +/- 0.0000 (spread sur seeds = 0.0000)
Rule   0 (classe I     ), n_cells= 64: asymptotique = 0.0000 +/- 0.0000 (spread sur seeds = 0.0000)
Rule   0 (classe I     ), n_cells=128: asymptotique = 0.0000 +/- 0.0000 (spread sur seeds = 0.0000)
Rule   4 (classe II    ), n_cells= 16: asymptotique = 0.0417 +/- 0.0722 (spread sur seeds = 0.1250)
Rule   4 (classe II    ), n_cells= 32: asymptotique = 0.1042 +/- 0.0477 (spread sur seeds = 0.0938)
Rule   4 (classe II    ), n_cells= 64: asymptotique = 0.1146 +/- 0.0549 (spread sur seeds = 0.1094)
Rule   4 (classe II    ), n_cells=128: asymptotique = 0.1198 +/- 0.0197 (spread sur seeds = 0.0391)
Rule  30 (classe III   ), n_cells= 16: asymptotique = 0.4688 +/- 0.0000 (spread sur seeds = 0.0000)
Rule  30 (classe III   ), n_cells= 32: asymptotique = 0.4844 +/- 0.0000 (spread sur seeds = 0.0000)
